# ⚡ TensorFlow 2.x Deep Learning Curriculum
## Low-Level Mastery: GradientTape, tf.function, tf.data, Distribution Strategies

> **Framework:** TensorFlow 2.x (native ops + Keras integration)  
> **Level:** Intermediate → Advanced → Production  
> **Estimated Time:** 60–80 hours  

---

## 📋 Table of Contents

1. [TensorFlow Fundamentals](#1-tf-fundamentals)
2. [GradientTape — Custom Training Loops](#2-gradienttape)
3. [tf.function & AutoGraph](#3-tf-function)
4. [tf.data Advanced Pipelines](#4-tf-data)
5. [Custom Training from Scratch](#5-custom-training)
6. [CNN Image Classification](#6-cnn)
7. [Recurrent Networks & Seq2Seq](#7-rnn-seq2seq)
8. [BERT Fine-Tuning with TF-Hub](#8-bert)
9. [Distribution Strategies — Multi-GPU](#9-distribution)
10. [TensorBoard & Profiling](#10-tensorboard)
11. [TF Serving & SavedModel](#11-serving)
12. [Advanced: Custom Ops & XLA](#12-advanced)
13. [Capstone Projects](#13-capstone)


---
## 1. TensorFlow Fundamentals
### ⏱ Estimated Time: 2 hours


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
matplotlib.rcParams['figure.dpi'] = 120

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

# Memory growth — prevents TF from grabbing all VRAM
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
# ── Tensor operations ────────────────────────────────────────────────────────
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.Variable([[0.5, 0.5], [0.5, 0.5]])

print('a dtype:', a.dtype, '  shape:', a.shape)
print('b dtype:', b.dtype, '  shape:', b.shape)

# Broadcasting
print('\na + 10       :\n', (a + 10).numpy())
print('a @ b        :\n', tf.matmul(a, b).numpy())
print('element-wise :\n', (a * b).numpy())
print('reduce_sum   :', tf.reduce_sum(a).numpy())
print('reduce_mean  :', tf.reduce_mean(a).numpy())

# tf.Variable is mutable
b.assign_add(tf.ones_like(b) * 0.1)
print('\nb after assign_add:\n', b.numpy())


In [ ]:
# ── Automatic differentiation primer ─────────────────────────────────────────
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x**3 + 2*x**2 - x + 1   # dy/dx = 3x² + 4x - 1

dy_dx = tape.gradient(y, x)
expected = 3*9 + 4*3 - 1  # = 38
print(f'dy/dx at x=3 → computed: {dy_dx.numpy():.1f}  expected: {expected}')

# Higher-order derivatives
with tf.GradientTape() as t2:
    with tf.GradientTape() as t1:
        y = x**3
    dy  = t1.gradient(y, x)   # 3x²
d2y = t2.gradient(dy, x)      # 6x
print(f'd²y/dx² at x=3 → computed: {d2y.numpy():.1f}  expected: {6*3}')


---
## 2. GradientTape — Custom Training Loops
### ⏱ Estimated Time: 4 hours

GradientTape records operations for automatic differentiation.  
Use it when you need **full control** over the training process.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Data
X, y = make_classification(n_samples=5000, n_features=20, n_informative=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype('float32')
X_test  = scaler.transform(X_test).astype('float32')
y_train = y_train.astype('float32')
y_test  = y_test.astype('float32')

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
optimizer   = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_fn     = tf.keras.losses.BinaryCrossentropy()
train_acc   = tf.keras.metrics.BinaryAccuracy()
val_acc     = tf.keras.metrics.BinaryAccuracy()
train_loss  = tf.keras.metrics.Mean()
val_loss    = tf.keras.metrics.Mean()

print('Model and metrics ready.')


In [ ]:
# ── Single training & validation steps ───────────────────────────────────────
@tf.function   # compile to graph for speed
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss   = loss_fn(y_batch, logits)
        loss  += sum(model.losses)   # regularisation losses
    grads = tape.gradient(loss, model.trainable_variables)
    # Gradient clipping — essential for deep networks
    grads, global_norm = tf.clip_by_global_norm(grads, clip_norm=1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_loss.update_state(loss)
    train_acc.update_state(y_batch, logits)
    return global_norm

@tf.function
def val_step(x_batch, y_batch):
    logits = model(x_batch, training=False)
    loss   = loss_fn(y_batch, logits)
    val_loss.update_state(loss)
    val_acc.update_state(y_batch, logits)

# ── Full training loop ────────────────────────────────────────────────────────
EPOCHS     = 40
BATCH_SIZE = 128
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(4000).batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    for xb, yb in train_ds:
        train_step(xb, yb)
    for xb, yb in val_ds:
        val_step(xb, yb)

    tl = train_loss.result().numpy(); ta = train_acc.result().numpy()
    vl = val_loss.result().numpy();   va = val_acc.result().numpy()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)

    if va > best_val_acc:
        best_val_acc = va
        model.save_weights('/tmp/best_weights.tf')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} — '
              f'loss: {tl:.4f}  acc: {ta:.4f} | '
              f'val_loss: {vl:.4f}  val_acc: {va:.4f}')

    for m in [train_loss, train_acc, val_loss, val_acc]:
        m.reset_state()

model.load_weights('/tmp/best_weights.tf')
print(f'\nBest val_acc: {best_val_acc:.4f}')


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history['train_loss'], label='Train', lw=2)
ax1.plot(history['val_loss'],   label='Val',   lw=2, ls='--')
ax1.set_title('Loss', fontweight='bold'); ax1.legend()
ax2.plot(history['train_acc'], label='Train', lw=2)
ax2.plot(history['val_acc'],   label='Val',   lw=2, ls='--')
ax2.set_title('Accuracy', fontweight='bold'); ax2.legend()
plt.tight_layout(); plt.show()


---
## 3. tf.function & AutoGraph
### ⏱ Estimated Time: 2 hours

`@tf.function` traces Python code into a **TensorFlow graph** — typically 2–10× faster.

| Mode | Speed | Flexibility |
|---|---|---|
| Eager (default) | Baseline | Full Python |
| `@tf.function` | 2–10× faster | Pure TF ops only |
| `jit_compile=True` (XLA) | Up to 3× more | Strict subset |


In [ ]:
# ── Speed comparison: eager vs graph ─────────────────────────────────────────
def matmul_heavy(x):
    for _ in range(20):
        x = tf.matmul(x, x)
    return x

@tf.function
def matmul_heavy_graph(x):
    for _ in range(20):
        x = tf.matmul(x, x)
    return x

@tf.function(jit_compile=True)  # XLA compilation
def matmul_heavy_xla(x):
    for _ in range(20):
        x = tf.matmul(x, x)
    return x

x = tf.random.normal([256, 256])

# Warm up
matmul_heavy_graph(x); matmul_heavy_xla(x)

N = 50
t0 = time.perf_counter()
for _ in range(N): matmul_heavy(x)
t_eager = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N): matmul_heavy_graph(x)
t_graph = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N): matmul_heavy_xla(x)
t_xla = (time.perf_counter() - t0) / N * 1000

print(f'Eager:     {t_eager:.2f} ms')
print(f'@tf.function: {t_graph:.2f} ms  ({t_eager/t_graph:.1f}× speedup)')
print(f'XLA:       {t_xla:.2f} ms  ({t_eager/t_xla:.1f}× speedup)')


In [ ]:
# ── AutoGraph: Python control flow inside @tf.function ───────────────────────
@tf.function
def compute_with_branches(x, threshold=0.5):
    """AutoGraph converts Python if/for/while to tf.cond/tf.while_loop."""
    # Python for → tf.while_loop
    result = tf.zeros_like(x)
    for i in tf.range(5):
        # Python if → tf.cond
        multiplier = tf.cond(
            tf.cast(i, tf.float32) > threshold * 5,
            lambda: 2.0, lambda: 0.5
        )
        result = result + x * multiplier
    return result

x_test = tf.constant([1.0, 2.0, 3.0])
print('Result:', compute_with_branches(x_test).numpy())

# Inspect the concrete function / graph
cf = compute_with_branches.get_concrete_function(x_test)
print('Concrete function signature:', cf.structured_input_signature)


In [ ]:
# ── tf.function tracing gotchas ──────────────────────────────────────────────
trace_count = 0

@tf.function
def count_traces(x):
    global trace_count
    trace_count += 1
    return x * 2

count_traces(tf.constant(1.0))   # trace 1: float32 scalar
count_traces(tf.constant(1.0))   # no retrace — same signature
count_traces(tf.constant(1))     # trace 2: int32 scalar (different dtype!)
count_traces(tf.constant([1.0, 2.0]))  # trace 3: float32 vector
print(f'Total traces: {trace_count}')  # should be 3
print()
print('⚠ Gotcha: Python scalars always retrace!')
print('   Bad:  count_traces(1.0)  → retraces every call')
print('   Good: count_traces(tf.constant(1.0))  → traces once')


---
## 4. tf.data Advanced Pipelines
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── TFRecord — the preferred storage format ───────────────────────────────────
import os

def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def _float_feature(values):
    return tf.train.Feature(float_list=tf.train.FloatList(value=values))

def serialize_example(features, label):
    feat_dict = {
        'features': _float_feature(features),
        'label':    _int64_feature(label)
    }
    proto = tf.train.Example(features=tf.train.Features(feature=feat_dict))
    return proto.SerializeToString()

# Write a small TFRecord file
TFRECORD_PATH = '/tmp/demo.tfrecord'
with tf.io.TFRecordWriter(TFRECORD_PATH) as writer:
    for i in range(1000):
        feats = np.random.randn(20).astype('float32')
        lbl   = np.random.randint(0, 2)
        writer.write(serialize_example(feats, lbl))
print(f'Written: {os.path.getsize(TFRECORD_PATH) / 1024:.1f} KB')

# Read back
FEAT_SPEC = {
    'features': tf.io.FixedLenFeature([20], tf.float32),
    'label':    tf.io.FixedLenFeature([],   tf.int64),
}
def parse_fn(serialized):
    parsed = tf.io.parse_single_example(serialized, FEAT_SPEC)
    return parsed['features'], tf.cast(parsed['label'], tf.float32)

tfrecord_ds = (
    tf.data.TFRecordDataset(TFRECORD_PATH)
    .map(parse_fn, num_parallel_calls=AUTOTUNE)
    .shuffle(500)
    .batch(64)
    .prefetch(AUTOTUNE)
)
batch = next(iter(tfrecord_ds))
print('Batch from TFRecord → X:', batch[0].shape, '  y:', batch[1].shape)


In [ ]:
# ── Pipeline performance benchmark ───────────────────────────────────────────
def benchmark_pipeline(dataset, n_batches=100):
    start = time.perf_counter()
    for i, _ in enumerate(dataset):
        if i >= n_batches:
            break
    return (time.perf_counter() - start) / n_batches * 1000

# Naive pipeline
naive = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .batch(128)
)
# Optimised pipeline
optimised = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(4000)
    .batch(128)
    .cache()
    .prefetch(AUTOTUNE)
)
t_naive = benchmark_pipeline(naive)
t_opt   = benchmark_pipeline(optimised)
print(f'Naive pipeline    : {t_naive:.3f} ms/batch')
print(f'Optimised pipeline: {t_opt:.3f} ms/batch')
print(f'Speedup           : {t_naive/t_opt:.1f}×')


---
## 5. Custom Training from Scratch
### ⏱ Estimated Time: 4 hours

### 🎯 Project: ResNet-style network trained with a custom loop, LR warmup, and gradient accumulation


In [ ]:
class ConvBNReLU(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size=3, strides=1, **kw):
        super().__init__(**kw)
        self.conv = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides,
                                            padding='same', use_bias=False)
        self.bn   = tf.keras.layers.BatchNormalization()

    def call(self, x, training=False):
        return tf.nn.relu(self.bn(self.conv(x), training=training))


class ResBlock(tf.keras.layers.Layer):
    def __init__(self, filters, strides=1, **kw):
        super().__init__(**kw)
        self.c1  = ConvBNReLU(filters, strides=strides)
        self.c2  = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.shortcut = tf.keras.layers.Conv2D(filters, 1, strides=strides,
                                                padding='same', use_bias=False)

    def call(self, x, training=False):
        h = self.c1(x, training=training)
        h = self.bn2(self.c2(h), training=training)
        return tf.nn.relu(h + self.shortcut(x))


class MiniResNet(tf.keras.Model):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem   = ConvBNReLU(64, 3)
        self.layer1 = ResBlock(64)
        self.layer2 = ResBlock(128, strides=2)
        self.layer3 = ResBlock(256, strides=2)
        self.pool   = tf.keras.layers.GlobalAveragePooling2D()
        self.drop   = tf.keras.layers.Dropout(0.4)
        self.fc     = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        x = self.stem(x,   training=training)
        x = self.layer1(x, training=training)
        x = self.layer2(x, training=training)
        x = self.layer3(x, training=training)
        x = self.pool(x)
        x = self.drop(x, training=training)
        return self.fc(x)

print('MiniResNet defined.')
dummy = tf.random.normal([2, 32, 32, 3])
net   = MiniResNet(10)
out   = net(dummy, training=True)
print('Output shape:', out.shape)
print('Parameters  :', net.count_params())


In [ ]:
# ── Gradient accumulation training loop ──────────────────────────────────────
(x_train_c, y_train_c), (x_test_c, y_test_c) = tf.keras.datasets.cifar10.load_data()
y_train_c = y_train_c.flatten(); y_test_c = y_test_c.flatten()

def preprocess(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.1)
    x = tf.clip_by_value(x, 0, 1)
    return x, tf.cast(y, tf.int32)

cifar_train = (
    tf.data.Dataset.from_tensor_slices((x_train_c, y_train_c))
    .shuffle(50000).map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(128).prefetch(AUTOTUNE)
)
cifar_val = (
    tf.data.Dataset.from_tensor_slices((x_test_c, y_test_c))
    .map(lambda x, y: (tf.cast(x, tf.float32)/255.0, tf.cast(y, tf.int32)))
    .batch(256).prefetch(AUTOTUNE)
)

resnet_model = MiniResNet(10)
optimizer_r  = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9, nesterov=True)
loss_fn_r    = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
ACCUM_STEPS  = 2   # effective batch = 128 × 2 = 256

@tf.function
def train_step_accum(batches):
    """Gradient accumulation over multiple mini-batches."""
    accum_grads = [tf.zeros_like(v) for v in resnet_model.trainable_variables]
    total_loss  = tf.constant(0.0)
    for x_b, y_b in batches:
        with tf.GradientTape() as tape:
            logits = resnet_model(x_b, training=True)
            loss   = loss_fn_r(y_b, logits)
        grads = tape.gradient(loss, resnet_model.trainable_variables)
        accum_grads = [ag + g for ag, g in zip(accum_grads, grads)]
        total_loss += loss
    # Average and apply
    avg_grads = [g / tf.cast(ACCUM_STEPS, tf.float32) for g in accum_grads]
    avg_grads, _ = tf.clip_by_global_norm(avg_grads, 1.0)
    optimizer_r.apply_gradients(zip(avg_grads, resnet_model.trainable_variables))
    return total_loss / ACCUM_STEPS

# LR schedule: cosine decay
def cosine_lr(epoch, total=50, base=0.1, min_lr=1e-5):
    return min_lr + 0.5 * (base - min_lr) * (1 + np.cos(np.pi * epoch / total))

print('Training MiniResNet with gradient accumulation...')
for epoch in range(20):
    lr = cosine_lr(epoch, total=20)
    optimizer_r.learning_rate.assign(lr)

    batch_iter = iter(cifar_train)
    epoch_loss = []
    while True:
        try:
            batch_group = [next(batch_iter) for _ in range(ACCUM_STEPS)]
            loss = train_step_accum(batch_group)
            epoch_loss.append(loss.numpy())
        except StopIteration:
            break

    # Validation accuracy
    correct = total_n = 0
    for xv, yv in cifar_val:
        preds = tf.argmax(resnet_model(xv, training=False), axis=1)
        correct  += tf.reduce_sum(tf.cast(preds == tf.cast(yv, tf.int64), tf.int32)).numpy()
        total_n  += len(yv)
    val_acc_r = correct / total_n

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/20  lr={lr:.5f}  '
              f'loss={np.mean(epoch_loss):.4f}  val_acc={val_acc_r:.4f}')


---
## 6. CNN Image Classification with tf.keras Applications
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── EfficientNetV2 with Feature Extraction + Fine-tuning ─────────────────────
base_model = tf.keras.applications.EfficientNetV2S(
    input_shape=(96, 96, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs  = tf.keras.Input((32, 32, 3))
x  = tf.keras.layers.Resizing(96, 96)(inputs)
x  = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x  = base_model(x, training=False)
x  = tf.keras.layers.GlobalAveragePooling2D()(x)
x  = tf.keras.layers.Dense(256, activation='relu')(x)
x  = tf.keras.layers.Dropout(0.4)(x)
out= tf.keras.layers.Dense(10, activation='softmax')(x)

effnet = tf.keras.Model(inputs, out)
effnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
print(f'Trainable params: {effnet.count_params():,}')

# Feature extraction phase
effnet.fit(cifar_train, validation_data=cifar_val, epochs=5, verbose=1)

# Fine-tuning: unfreeze last 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
effnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
effnet.fit(cifar_train, validation_data=cifar_val, epochs=10, verbose=1)


---
## 7. Recurrent Networks & Sequence-to-Sequence
### ⏱ Estimated Time: 6 hours

### 🎯 Project: Character-Level Seq2Seq Text Generation


In [ ]:
# ── Prepare character-level dataset ──────────────────────────────────────────
text = """To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them to die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished to die to sleep
To sleep perchance to dream ay there's the rub"""

chars   = sorted(set(text))
VOCAB   = len(chars)
c2i     = {c: i for i, c in enumerate(chars)}
i2c     = {i: c for c, i in c2i.items()}
encoded = [c2i[c] for c in text]

SEQ_LEN = 40
STEP    = 3
seqs, nexts = [], []
for i in range(0, len(encoded) - SEQ_LEN, STEP):
    seqs.append(encoded[i:i+SEQ_LEN])
    nexts.append(encoded[i+SEQ_LEN])

X_seq = tf.keras.utils.to_categorical(seqs,  VOCAB).astype('float32')
y_seq = tf.keras.utils.to_categorical(nexts, VOCAB).astype('float32')
print(f'Vocab size: {VOCAB}  Sequences: {len(seqs)}  X: {X_seq.shape}')


In [ ]:
# ── Stacked LSTM language model ───────────────────────────────────────────────
lm = tf.keras.Sequential([
    tf.keras.layers.LSTM(256, return_sequences=True, input_shape=(SEQ_LEN, VOCAB)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(256),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(VOCAB, activation='softmax')
])
lm.compile(optimizer=tf.keras.optimizers.RMSprop(1e-3),
            loss='categorical_crossentropy')
lm.fit(X_seq, y_seq, batch_size=64, epochs=60, verbose=0)
print('LSTM language model trained.')

def sample_with_temperature(preds, temperature=1.0):
    preds  = np.log(preds + 1e-8) / temperature
    preds  = np.exp(preds) / np.sum(np.exp(preds))
    return np.random.choice(len(preds), p=preds)

def generate_text(seed, n_chars=200, temp=0.8):
    generated = seed
    seq       = [c2i.get(c, 0) for c in seed[-SEQ_LEN:]]
    seq       = seq + [0] * max(0, SEQ_LEN - len(seq))
    for _ in range(n_chars):
        x_in = tf.keras.utils.to_categorical([seq], VOCAB).astype('float32')
        pred = lm.predict(x_in, verbose=0)[0]
        idx  = sample_with_temperature(pred, temp)
        generated += i2c[idx]
        seq = seq[1:] + [idx]
    return generated

print('Generated text (temperature=0.8):')
print(generate_text('to be or not ', n_chars=150))


---
## 8. BERT Fine-Tuning with TF-Hub
### ⏱ Estimated Time: 6 hours

### 🎯 Project: Sentence-Pair Classification (paraphrase detection)


In [ ]:
# ── BERT fine-tuning template (requires internet for TF-Hub download) ─────────
BERT_CODE = '''
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text  # required for BERT preprocessing

BERT_MODEL_URL = "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1"
PREPROCESS_URL = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"

def build_bert_classifier(num_labels=2, dropout=0.1):
    text_input  = tf.keras.Input(shape=(), dtype=tf.string, name="text")
    preprocessor= hub.KerasLayer(PREPROCESS_URL, name="preprocessing")
    encoder     = hub.KerasLayer(BERT_MODEL_URL, trainable=True, name="bert_encoder")

    enc_inputs  = preprocessor(text_input)
    enc_outputs = encoder(enc_inputs)
    pooled      = enc_outputs["pooled_output"]     # [CLS] token
    pooled      = tf.keras.layers.Dropout(dropout)(pooled)
    logits      = tf.keras.layers.Dense(num_labels, activation="softmax")(pooled)

    return tf.keras.Model(text_input, logits, name="BERT_Classifier")

model = build_bert_classifier()
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5, epsilon=1e-8),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

# Fine-tuning with GLUE MRPC dataset
import tensorflow_datasets as tfds
ds = tfds.load("glue/mrpc", split=["train", "validation"])
# ... preprocessing and training here ...
'''
print('BERT Fine-Tuning Template:')
print(BERT_CODE)
with open('/tmp/bert_finetune.py', 'w') as f:
    f.write(BERT_CODE.strip())
print('\nSaved to /tmp/bert_finetune.py')


---
## 9. Distribution Strategies — Multi-GPU Training
### ⏱ Estimated Time: 3 hours

| Strategy | Use Case |
|---|---|
| `MirroredStrategy` | Multiple GPUs, single machine |
| `MultiWorkerMirroredStrategy` | Multiple machines |
| `TPUStrategy` | Google TPU pods |
| `ParameterServerStrategy` | Async, large-scale |


In [ ]:
# ── MirroredStrategy template ─────────────────────────────────────────────────
strategy = tf.distribute.MirroredStrategy()
print(f'Number of replicas: {strategy.num_replicas_in_sync}')

# All model creation & compilation MUST happen inside strategy.scope()
with strategy.scope():
    dist_model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(20,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    dist_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3 * strategy.num_replicas_in_sync),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

# Scale batch size with number of replicas
GLOBAL_BATCH = 128 * strategy.num_replicas_in_sync
dist_train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(4000)
    .batch(GLOBAL_BATCH)
    .prefetch(AUTOTUNE)
)
dist_model.fit(dist_train_ds, epochs=10, verbose=1)
print(f'\nDistributed training complete. Effective batch size: {GLOBAL_BATCH}')


In [ ]:
# ── Custom distributed training step ─────────────────────────────────────────
with strategy.scope():
    dist_opt    = tf.keras.optimizers.Adam(1e-3)
    dist_loss_fn= tf.keras.losses.BinaryCrossentropy(
        from_logits=False,
        reduction=tf.keras.losses.Reduction.NONE   # important: no auto-reduce
    )

def compute_loss(labels, predictions):
    per_example_loss = dist_loss_fn(labels, predictions)
    # Average across replicas
    return tf.nn.compute_average_loss(per_example_loss,
                                       global_batch_size=GLOBAL_BATCH)

@tf.function
def distributed_train_step(dataset_inputs):
    def step_fn(inputs):
        x, y = inputs
        with tf.GradientTape() as tape:
            logits = dist_model(x, training=True)
            loss   = compute_loss(y, logits)
        grads = tape.gradient(loss, dist_model.trainable_variables)
        dist_opt.apply_gradients(zip(grads, dist_model.trainable_variables))
        return loss
    per_replica_losses = strategy.run(step_fn, args=(dataset_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

print('Distributed training step compiled ✓')
print('Pattern: strategy.run() → strategy.reduce()')


---
## 10. TensorBoard & Profiling
### ⏱ Estimated Time: 2 hours


In [ ]:
import datetime

LOG_DIR = '/tmp/tb_logs/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

tb_model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(64,  activation='relu'),
    tf.keras.layers.Dense(1,   activation='sigmoid')
])
tb_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

tb_callbacks = [
    tf.keras.callbacks.TensorBoard(
        log_dir=LOG_DIR,
        histogram_freq=1,       # weight histograms every epoch
        write_graph=True,
        write_images=True,
        update_freq='epoch',
        profile_batch='5,10'    # profile batches 5–10
    )
]
tb_model.fit(X_train, y_train, validation_split=0.2,
              epochs=10, batch_size=128, callbacks=tb_callbacks, verbose=0)

print(f'TensorBoard logs: {LOG_DIR}')
print('To view: tensorboard --logdir', LOG_DIR)

# Custom scalar logging
writer = tf.summary.create_file_writer(LOG_DIR + '/custom')
with writer.as_default():
    for step, lr in enumerate([1e-3, 5e-4, 1e-4, 5e-5]):
        tf.summary.scalar('custom/learning_rate', lr, step=step)
        tf.summary.scalar('custom/fake_metric', np.sin(step), step=step)
print('Custom scalars written ✓')


---
## 11. TF Serving & SavedModel
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Export with custom serving signature ─────────────────────────────────────
class CIFARPredictor(tf.keras.Model):
    """Model with built-in preprocessing for serving."""
    CLASSES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    @tf.function(input_signature=[tf.TensorSpec([None, 32, 32, 3], tf.uint8)])
    def serve(self, raw_images):
        x = tf.cast(raw_images, tf.float32) / 255.0
        logits = self.base(x, training=False)
        classes= tf.argmax(logits, axis=1)
        confidence = tf.reduce_max(tf.nn.softmax(logits), axis=1)
        return {'class_id': classes, 'confidence': confidence}

predictor = CIFARPredictor(resnet_model)

EXPORT_PATH = '/tmp/tf_serving_model/1'   # versioned directory
tf.saved_model.save(
    predictor, EXPORT_PATH,
    signatures={'serving_default': predictor.serve}
)
print(f'SavedModel exported: {EXPORT_PATH}')

# Inspect the saved model
loaded = tf.saved_model.load(EXPORT_PATH)
infer  = loaded.signatures['serving_default']
sample = tf.cast(x_test_c[:4], tf.uint8)
result = infer(raw_images=sample)
print('\nServing output:')
for i in range(4):
    cls_id = result['class_id'][i].numpy()
    conf   = result['confidence'][i].numpy()
    true   = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck'][y_test_c[i]]
    print(f'  Sample {i}: predicted={CIFARPredictor.CLASSES[cls_id]:12s}  conf={conf:.3f}  true={true}')


In [ ]:
# ── TF Serving Docker command ─────────────────────────────────────────────────
docker_cmd = f"""
# Pull & start TF Serving
docker pull tensorflow/serving

docker run -d \\
  -p 8501:8501 \\
  -v /tmp/tf_serving_model:/models/cifar_classifier \\
  -e MODEL_NAME=cifar_classifier \\
  --name tf_serving \\
  tensorflow/serving

# REST API call (once server is running)
curl -X POST http://localhost:8501/v1/models/cifar_classifier:predict \\
  -d '{{"instances": [...]}}'

# Check model metadata
curl http://localhost:8501/v1/models/cifar_classifier
"""
print(docker_cmd)


---
## 12. Advanced: Custom Ops & XLA
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Custom gradient (straight-through estimator example) ─────────────────────
@tf.custom_gradient
def hard_sigmoid(x):
    """Forward: step function. Backward: straight-through (identity)."""
    def grad(dy):
        return dy  # straight-through: pass gradient unchanged
    return tf.cast(x > 0, tf.float32), grad

x_ste = tf.Variable([-1.0, 0.5, 2.0])
with tf.GradientTape() as tape:
    y_ste = tf.reduce_sum(hard_sigmoid(x_ste))
g = tape.gradient(y_ste, x_ste)
print('hard_sigmoid output:', hard_sigmoid(x_ste).numpy())
print('gradient (STE)      :', g.numpy())  # [1, 1, 1] — all ones


In [ ]:
# ── XLA compilation for maximum speed ────────────────────────────────────────
@tf.function(jit_compile=True)
def xla_dense_forward(W, b, x):
    """XLA-compiled matrix multiply + bias + activation."""
    return tf.nn.relu(tf.matmul(x, W) + b)

W = tf.random.normal([512, 512])
b = tf.zeros([512])
x = tf.random.normal([256, 512])

# Warm-up
xla_dense_forward(W, b, x)

N = 200
t0 = time.perf_counter()
for _ in range(N):
    _ = tf.matmul(x, W) + b
t_eager = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N):
    _ = xla_dense_forward(W, b, x)
t_xla = (time.perf_counter() - t0) / N * 1000

print(f'Eager: {t_eager:.3f} ms   XLA: {t_xla:.3f} ms   Speedup: {t_eager/t_xla:.2f}×')


---
## 13. Capstone Projects

### 🏆 Capstone A: Production Image Classifier
```
CIFAR-10/Custom data
→ tf.data + TFRecord pipeline
→ MiniResNet with cosine LR + gradient accumulation
→ MirroredStrategy multi-GPU
→ TensorBoard profiling
→ SavedModel with serving signature
→ TF Serving Docker container
```

### 🏆 Capstone B: BERT NLP Pipeline
```
GLUE benchmark dataset
→ TF-Hub BERT preprocessing
→ Fine-tune bert-base with AdamW + linear warmup
→ Eval: accuracy, F1, MCC
→ ONNX export for cross-framework serving
```

### 🏆 Capstone C: Real-Time Inference System
```
Train model → SavedModel → TFLite INT8 quant
→ FastAPI REST endpoint
→ Prometheus metrics + Grafana dashboard
→ Docker Compose stack
→ GitHub Actions CI/CD
```

---
## 📋 Quick Reference: TensorFlow Idioms

```python
# Always use tf.function for repeated compute
@tf.function
def step(x): ...

# Mixed precision
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Gradient clipping
grads, _ = tf.clip_by_global_norm(grads, 1.0)

# Profile
tf.profiler.experimental.start(log_dir)
# ... code ...
tf.profiler.experimental.stop()

# Check variables on device
for v in model.variables:
    print(v.device, v.name)
```

---
*TensorFlow Deep Learning Curriculum — Complete ✓*
